In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================================
# IMPORTS
# =========================================================

import torch
import torch.nn as nn
import librosa
from transformers import HubertModel, Wav2Vec2FeatureExtractor

# =========================================================
# CONFIG
# =========================================================

MODEL_PATH = "/content/drive/MyDrive/IIITH_Voice/best_emotion_model.pth"

TEST_AUDIO = "/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data/YAF_disgust/YAF_bar_disgust.wav"

"""Test with these:
/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data/OAF_Fear/OAF_bath_fear.wav
/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data/OAF_happy/OAF_back_happy.wav
/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data/OAF_angry/OAF_beg_angry.wav
/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data/OAF_Sad/OAF_beg_sad.wav
/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data/YAF_pleasant_surprised/YAF_base_ps.wav
/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data/YAF_happy/YAF_base_happy.wav
"""

MODEL_NAME = "facebook/hubert-base-ls960"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# =========================================================
# LABELS
# =========================================================

classes = [
    'angry',
    'disgust',
    'fear',
    'happy',
    'neutral',
    'pleasant_surprise',
    'sad'
]

# =========================================================
# LOAD HUBERT
# =========================================================

feature_extractor = (
    Wav2Vec2FeatureExtractor
    .from_pretrained(MODEL_NAME)
)

hubert = HubertModel.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True
).to(DEVICE)

hubert.eval()

# =========================================================
# ATTENTION
# =========================================================

class Attention(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Linear(
            hidden_size * 2,
            1
        )

    def forward(self, lstm_outputs, lengths):

        scores = self.attention(
            lstm_outputs
        ).squeeze(-1)

        max_len = lstm_outputs.size(1)

        mask = torch.arange(
            max_len,
            device=lengths.device
        )[None, :] < lengths[:, None]

        scores[~mask] = -1e9

        weights = torch.softmax(
            scores,
            dim=1
        )

        weights = weights.unsqueeze(-1)

        context = torch.sum(
            weights * lstm_outputs,
            dim=1
        )

        return context

# =========================================================
# MODEL
# =========================================================

class EmotionModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=128,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.attention = Attention(128)

        self.classifier = nn.Sequential(

            nn.Linear(256, 128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128, len(classes))
        )

    def forward(self, x, lengths):

        packed = nn.utils.rnn.pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_outputs, _ = self.lstm(packed)

        lstm_outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs,
            batch_first=True
        )

        attention_output = self.attention(
            lstm_outputs,
            lengths
        )

        logits = self.classifier(
            attention_output
        )

        return logits

# =========================================================
# LOAD MODEL
# =========================================================

model = EmotionModel().to(DEVICE)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=DEVICE
    )
)

model.eval()

# =========================================================
# FEATURE EXTRACTION
# =========================================================

def extract_hubert_sequence(path, sr=16000):

    waveform, sr = librosa.load(
        path,
        sr=sr,
        mono=True
    )

    waveform, _ = librosa.effects.trim(
        waveform,
        top_db=20
    )

    waveform = librosa.util.normalize(
        waveform
    )

    inputs = feature_extractor(
        waveform,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True
    )

    input_values = (
        inputs["input_values"]
        .to(DEVICE)
    )

    with torch.no_grad():

        outputs = hubert(input_values)

        hidden_states = outputs.hidden_states

    selected_layers = hidden_states[6:13]

    stacked_layers = torch.stack(
        selected_layers,
        dim=0
    )

    weighted_sum = stacked_layers.mean(dim=0)

    embedding = (
        weighted_sum
        .squeeze(0)
    )

    return embedding

# =========================================================
# PREDICTION
# =========================================================

embedding = extract_hubert_sequence(
    TEST_AUDIO
)

embedding = (
    embedding
    .unsqueeze(0)
    .to(DEVICE)
)

lengths = torch.tensor(
    [embedding.shape[1]]
).to(DEVICE)

with torch.no_grad():

    outputs = model(
        embedding,
        lengths
    )

    pred = torch.argmax(
        outputs,
        dim=1
    ).item()

predicted_emotion = classes[pred]

print(
    f"\nPredicted Emotion: "
    f"{predicted_emotion}"
)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]


Predicted Emotion: disgust
